# Phase 7 — Evaluation

Computes CLIP alignment, LPIPS diversity, style consistency, and manual ratings across sweep outputs. Run after a full sweep has completed in Colab.

In [ ]:
# Cell 1 — Setup (mount Drive, clone/pull repo, install deps, PYTHONPATH)
import os, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = Path("/content/roomify")
if not REPO_DIR.exists():
    !git clone https://github.com/ben-blake/roomify.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# Install packages absent from the Colab runtime (each notebook has its own runtime)
!pip install -q -r {REPO_DIR}/requirements.txt

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

OUTPUTS_DIR = Path("/content/drive/MyDrive/roomify/outputs")
print(f"Outputs dir: {OUTPUTS_DIR}  exists={OUTPUTS_DIR.exists()}")

In [ ]:
# Cell 2 — Pick sweep to evaluate
import ipywidgets as widgets
from IPython.display import display

sweeps = sorted(OUTPUTS_DIR.iterdir()) if OUTPUTS_DIR.exists() else []
opts = [s.name for s in sweeps if s.is_dir()]

w = widgets.Dropdown(options=opts, description="Sweep:")
display(w)

In [ ]:
# Cell 3 — CLIP alignment
from roomify.evaluation import clipAlignment

SWEEP = OUTPUTS_DIR / w.value
df_clip = clipAlignment(SWEEP)
print(f"Mean CLIP score: {df_clip['clipScore'].mean():.4f}")
df_clip.sort_values("clipScore", ascending=False).head(10)

In [ ]:
# Cell 4 — LPIPS diversity + style consistency
from roomify.evaluation import lpipsDiversity, styleConsistency

print(f"LPIPS diversity:   {lpipsDiversity(SWEEP):.4f}")
print(f"Style consistency: {styleConsistency(SWEEP):.4f}")

In [ ]:
# Cell 5 — Contact sheet
from roomify.reporting import contactSheet

sheet = contactSheet(SWEEP, thumbSize=256)
sheet.save(str(SWEEP / "contact_sheet.png"))
sheet

In [ ]:
# Cell 6 — Metrics markdown table
from roomify.reporting import metricsTable

from IPython.display import Markdown
Markdown(metricsTable(SWEEP))

In [ ]:
# Cell 7 — Controlled vs uncontrolled comparison
# Requires runs with controlled=True and controlled=False for the same spec+seed.
# The core.yaml sweep is uncontrolled-only; point SWEEP at a directory that
# contains both, or run Cell 9 of 00_launchColab.ipynb first.
import json
from PIL import Image
import matplotlib.pyplot as plt

pairs = []
runs = {}
for rj in sorted(SWEEP.rglob("run.json")):
    d = json.loads(rj.read_text())
    key = (d.get("spec", {}).get("id"), d.get("seed"))
    runs.setdefault(key, {})["ctrl" if d.get("controlled") else "unctrl"] = rj.parent

for key, v in runs.items():
    if "ctrl" in v and "unctrl" in v:
        pairs.append((key, v["unctrl"], v["ctrl"]))

if not pairs:
    print("No ctrl/unctrl pairs found in this sweep.")
    print("To generate pairs: run Cell 9 of 00_launchColab.ipynb, then re-point SWEEP.")
else:
    for (spec_id, seed), unctrl, ctrl in pairs[:3]:
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(Image.open(unctrl / "img_0.png")); axes[0].set_title("Uncontrolled"); axes[0].axis("off")
        axes[1].imshow(Image.open(ctrl / "img_0.png")); axes[1].set_title("Controlled (depth)"); axes[1].axis("off")
        fig.suptitle(f"{spec_id} seed={seed}")
        plt.tight_layout()
        plt.savefig(str(SWEEP / f"comparison_{spec_id}_s{seed}.png"), dpi=150)
        plt.show()
        print(f"Saved: comparison_{spec_id}_s{seed}.png")

In [ ]:
# Cell 7b — Cross-strategy comparison (minimal vs descriptive vs styleAnchored)
# Works with the core.yaml sweep.  Shows one spec across all 3 strategies for a fixed seed.
import json
from PIL import Image
import matplotlib.pyplot as plt

STRATEGIES = ["minimal", "descriptive", "styleAnchored"]
FIXED_SEED = 42

# Build lookup: (spec_id, strategy, seed) -> run_dir
lookup = {}
for rj in sorted(SWEEP.rglob("run.json")):
    d = json.loads(rj.read_text())
    key = (d.get("spec", {}).get("id"), d.get("strategy"), d.get("seed"))
    lookup[key] = rj.parent

specs_with_all = [
    spec_id for spec_id in set(k[0] for k in lookup)
    if all((spec_id, s, FIXED_SEED) in lookup for s in STRATEGIES)
]

for spec_id in sorted(specs_with_all)[:3]:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, strategy in zip(axes, STRATEGIES):
        img_path = lookup[(spec_id, strategy, FIXED_SEED)] / "img_0.png"
        ax.imshow(Image.open(img_path))
        ax.set_title(strategy, fontsize=12)
        ax.axis("off")
    fig.suptitle(f"{spec_id}  seed={FIXED_SEED}", fontsize=14)
    plt.tight_layout()
    out = SWEEP / f"strategy_compare_{spec_id}_s{FIXED_SEED}.png"
    plt.savefig(str(out), dpi=150)
    plt.show()
    print(f"Saved: {out.name}")

In [ ]:
# Cell 8 — Export top-N images by CLIP score (slide-ready PNGs)
EXPORT_DIR = SWEEP / "top_exports"
EXPORT_DIR.mkdir(exist_ok=True)

top = df_clip.nlargest(6, "clipScore")
for _, row in top.iterrows():
    src = Path(row["imagePath"])
    if src.exists():
        dst = EXPORT_DIR / src.name
        Image.open(src).save(str(dst))
        print(f"Exported: {dst.name}  CLIP={row['clipScore']:.4f}")